# 06 - Pipeline Demo

End-to-end demonstration of the complete AI resume screening pipeline.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost spacy sentence-transformers joblib matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

In [ ]:
# Load trained models
models_dir = '../backend/app/ml_models'
os.makedirs(models_dir, exist_ok=True)

loaded_models = {}
for filename in ['ranking_model.pkl', 'scaler.pkl', 'skill_taxonomy.pkl']:
    path = os.path.join(models_dir, filename)
    if os.path.exists(path):
        loaded_models[filename] = joblib.load(path)
        print(f"Loaded: {filename}")
    else:
        print(f"Not found: {filename}")

ranking_model = loaded_models.get('ranking_model.pkl')
scaler = loaded_models.get('scaler.pkl')
skill_taxonomy = loaded_models.get('skill_taxonomy.pkl', {})

In [ ]:
# Load datasets
candidates = pd.read_csv('../datasets/candidates.csv')
jobs = pd.read_csv('../datasets/jobs.csv')

print(f"Candidates: {len(candidates)}, Jobs: {len(jobs)}")

In [ ]:
# Define the screening pipeline
def extract_skills(text, taxonomy=None):
    """Extract skills from text using the taxonomy."""
    if taxonomy is None:
        taxonomy = {}
    all_skills = []
    for cat_skills in taxonomy.values():
        all_skills.extend(cat_skills)
    
    text_lower = text.lower()
    found = [s for s in all_skills if s.lower() in text_lower]
    return list(set(found))

def calculate_skill_score(candidate_skills, required_skills):
    """Calculate skill match percentage."""
    if not required_skills:
        return 0.0
    cand_set = {s.lower() for s in candidate_skills}
    req_set = {s.lower() for s in required_skills}
    return (len(cand_set & req_set) / len(req_set)) * 100

def calculate_exp_score(candidate_exp, min_exp):
    """Calculate experience match score."""
    if min_exp == 0:
        return 100.0
    if candidate_exp >= min_exp:
        return min(100.0, 70 + (candidate_exp - min_exp) * 5)
    return max(0.0, (candidate_exp / min_exp) * 70)

def calculate_semantic_score(text, required_skills):
    """Keyword-based semantic similarity."""
    if not text or not required_skills:
        return 50.0
    text_lower = text.lower()
    matches = sum(1 for s in required_skills if s.lower() in text_lower)
    return min(100.0, (matches / len(required_skills)) * 80 + 20)

def detect_bias(text):
    """Simple bias detection."""
    masculine = {"aggressive", "dominant", "ninja", "rockstar", "guru", "alpha"}
    text_lower = text.lower()
    flagged = [t for t in masculine if t in text_lower]
    return len(flagged) > 0, flagged

print("Pipeline functions defined")

In [ ]:
# Screen candidates for a specific job
def screen_candidates(job_title, required_skills, min_experience, top_n=10):
    """Screen all candidates for a given job and return ranked results."""
    results = []
    
    for _, cand in candidates.iterrows():
        cand_skills = [s.strip() for s in cand['skills'].split(',')]
        
        skill_score = calculate_skill_score(cand_skills, required_skills)
        exp_score = calculate_exp_score(cand['experience'], min_experience)
        semantic_score = calculate_semantic_score(cand['resume_text'], required_skills)
        
        # Weighted overall score
        overall = skill_score * 0.4 + exp_score * 0.3 + semantic_score * 0.3
        
        # Bias check
        has_bias, flagged = detect_bias(cand['resume_text'])
        
        results.append({
            'name': cand['name'],
            'role': cand['role'],
            'experience': cand['experience'],
            'skill_score': round(skill_score, 1),
            'exp_score': round(exp_score, 1),
            'semantic_score': round(semantic_score, 1),
            'overall': round(overall, 1),
            'skills': cand_skills[:5],
            'has_bias': has_bias,
            'flagged_terms': flagged
        })
    
    # Sort by overall score
    results.sort(key=lambda x: x['overall'], reverse=True)
    return results[:top_n]

print("Screening function defined")

In [ ]:
# Demo: Screen for a Senior Frontend Engineer role
job_title = "Senior Frontend Engineer"
required_skills = ["React", "TypeScript", "JavaScript", "CSS", "Node.js", "Git"]
min_experience = 3

print(f"Job: {job_title}")
print(f"Required skills: {', '.join(required_skills)}")
print(f"Min experience: {min_experience} years")
print("\n" + "=" * 70)

results = screen_candidates(job_title, required_skills, min_experience, top_n=10)

print(f"{'Rank':<5} {'Name':<20} {'Overall':<10} {'Skill':<8} {'Exp':<8} {'Semantic':<10} {'Bias'}")
print("-" * 70)
for i, r in enumerate(results, 1):
    bias_icon = "⚠️" if r['has_bias'] else "✅"
    print(f"{i:<5} {r['name']:<20} {r['overall']:<10} {r['skill_score']:<8} {r['exp_score']:<8} {r['semantic_score']:<10} {bias_icon}")

In [ ]:
# Visualize top candidate scores
df_results = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Overall scores
axes[0].barh(range(len(df_results)), df_results['overall'], color='#2563eb')
axes[0].set_yticks(range(len(df_results)))
axes[0].set_yticklabels([r[:15] for r in df_results['name']])
axes[0].set_xlabel('Overall Score')
axes[0].set_title('Overall Scores')
axes[0].invert_yaxis()

# Score breakdown for top candidate
top = df_results.iloc[0]
axes[1].bar(['Skill', 'Experience', 'Semantic'], [top['skill_score'], top['exp_score'], top['semantic_score']],
            color=['#22c55e', '#f97316', '#8b5cf6'])
axes[1].set_ylabel('Score')
axes[1].set_title(f"Score Breakdown: {top['name'][:15]}")
axes[1].set_ylim(0, 100)

# Score distribution
axes[2].hist(df_results['overall'], bins=10, color='#2563eb', alpha=0.7, edgecolor='white')
axes[2].axvline(x=df_results['overall'].mean(), color='red', linestyle='--', label=f"Avg: {df_results['overall'].mean():.1f}")
axes[2].set_xlabel('Overall Score')
axes[2].set_ylabel('Count')
axes[2].set_title('Score Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Demo: Screen for a Data Scientist role
job_title = "Data Scientist"
required_skills = ["Python", "Machine Learning", "PyTorch", "Pandas", "SQL", "Statistics"]
min_experience = 2

print(f"\nJob: {job_title}")
print(f"Required skills: {', '.join(required_skills)}")
print("\n" + "=" * 70)

results_ds = screen_candidates(job_title, required_skills, min_experience, top_n=5)

for i, r in enumerate(results_ds, 1):
    bias_icon = "⚠️" if r['has_bias'] else "✅"
    print(f"{i}. {r['name']} (Score: {r['overall']}) {bias_icon}")
    print(f"   Skills: {', '.join(r['skills'])}")
    print(f"   Breakdown - Skill: {r['skill_score']}, Exp: {r['exp_score']}, Semantic: {r['semantic_score']}")
    print()

In [ ]:
# Pipeline summary
print("\n" + "=" * 50)
print("PIPELINE SUMMARY")
print("=" * 50)
print(f"Total candidates in pool: {len(candidates)}")
print(f"Total jobs available: {len(jobs)}")
print(f"Skills taxonomy categories: {len(skill_taxonomy)}")
print(f"Ranking model loaded: {ranking_model is not None}")
print(f"Scaler loaded: {scaler is not None}")
print("\nPipeline ready for production use!")